# 10.5 Bayesian Optimization

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.05_bayesian-optimization.ipynb)

Companion code for Section 10.5 of the book. Run the cells in order from top to bottom.

**Example 10.5.1**

In [ ]:
# pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import time
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Optuna: define the objective function ─────────────────────────
def objective_rf(trial):
    '''Objective function for Random Forest hyperparameter optimization.'''
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 500),
        'max_depth':         trial.suggest_categorical('max_depth', [None,3,5,7,10,15]),
        'max_features':      trial.suggest_float('max_features', 0.1, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'bootstrap':         trial.suggest_categorical('bootstrap', [True, False]),
    }
    model  = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()


def objective_gb(trial):
    '''Objective function for Gradient Boosting.'''
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 50, 400),
        'learning_rate':   trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'max_depth':       trial.suggest_int('max_depth', 2, 8),
        'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
        'max_features':    trial.suggest_float('max_features', 0.3, 1.0),
        'min_samples_leaf':trial.suggest_int('min_samples_leaf', 1, 10),
    }
    model  = GradientBoostingClassifier(**params, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()


# ── Run Optuna studies ─────────────────────────────────────────────
print('Running Optuna Bayesian Optimization...')

# Random Forest
t0 = time.time()
study_rf = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=50, n_jobs=-1)
t_rf = time.time() - t0

print(f'\nRandom Forest Optuna (50 trials):')
print(f'  Best AUC:    {study_rf.best_value:.4f}')
print(f'  Time:        {t_rf:.1f}s')
print(f'  Best params: {study_rf.best_params}')

# Gradient Boosting
t0 = time.time()
study_gb = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_gb.optimize(objective_gb, n_trials=50, n_jobs=-1)
t_gb = time.time() - t0
print(f'\nGradient Boosting Optuna (50 trials):')
print(f'  Best AUC:    {study_gb.best_value:.4f}')
print(f'  Time:        {t_gb:.1f}s')

# ── Compare: Grid vs Random vs Bayesian ──────────────────────────
print('\n=== Search Strategy Comparison ===')
print(f'{'Method':<28} {'Best AUC':>10} {'Trials':>8} {'Time(s)':>10}')
print('='*62)
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from scipy.stats import randint, uniform

# Grid Search (36 configs)
t0 = time.time()
gs_small = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {'n_estimators':[100,200,300],'max_depth':[None,5,10],'max_features':['sqrt',0.3,0.5,0.8]},
    cv=5, scoring='roc_auc', n_jobs=-1
)
gs_small.fit(X, y)
t_gs = time.time() - t0
print(f'{'Grid Search (36 configs)':<28} {gs_small.best_score_:>10.4f} {36:>8} {t_gs:>10.2f}')

# Random Search (50 trials)
t0 = time.time()
rs_50 = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {'n_estimators':randint(50,500),'max_depth':[None,3,5,7,10,15],
     'max_features':uniform(0.1,0.9),'min_samples_leaf':randint(1,10)},
    n_iter=50, cv=5, scoring='roc_auc', n_jobs=-1, random_state=42
)
rs_50.fit(X, y)
t_rs = time.time() - t0
print(f'{'Random Search (50 trials)':<28} {rs_50.best_score_:>10.4f} {50:>8} {t_rs:>10.2f}')
print(f'{'Bayesian Optuna RF (50 trials)':<28} {study_rf.best_value:>10.4f} {50:>8} {t_rf:>10.2f}')
print(f'{'Bayesian Optuna GB (50 trials)':<28} {study_gb.best_value:>10.4f} {50:>8} {t_gb:>10.2f}')

# ── Visualise Optuna results ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Bayesian Optimization - Optuna Analysis', fontsize=14, fontweight='bold')

# Optimization history
ax = axes[0]
trial_values_rf = [t.value for t in study_rf.trials]
best_so_far_rf  = np.maximum.accumulate(trial_values_rf)
ax.plot(trial_values_rf, 'b.', ms=6, alpha=0.5, label='Individual trial')
ax.plot(best_so_far_rf,  'r-', lw=2.5, label='Best so far')
ax.set_xlabel('Trial Number'); ax.set_ylabel('CV AUC')
ax.set_title('Optuna Optimization History (RF)\n(improvement concentrates early)')
ax.legend(); ax.grid(alpha=0.3)

# Parameter importance
ax = axes[1]
importances = optuna.importance.get_param_importances(study_rf)
ax.barh(list(importances.keys()), list(importances.values()),
       color='#FF6F00', edgecolor='white')
ax.set_xlabel('Hyperparameter Importance'); ax.set_title('Optuna Parameter Importance\n(which params matter most?)')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('bayesian_optimization.png', dpi=150, bbox_inches='tight')
plt.show()